In [ ]:
!pip install emlearn

In [ ]:
import numpy as np
import tensorflow as tf
from scipy import signal
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import os
import json # Added for JSON parsing
import re   # Added for regex to extract label from filename
import traceback

try:
    import emlearn
except ImportError:
    emlearn = None
    print("Import Emlearn failed !")

from google.colab import drive
drive.mount('/content/drive')

print(tf.__version__)
print("TensorFlow Keras version:", tf.keras.__version__)

import scipy
print("SciPy version:", scipy.__version__)

In [ ]:
# ============================================================
# 1. DSP FEATURE EXTRACTION (mirrors C++ spectral v4)
# ============================================================

CONFIG = {
    'axes': 3,
    'scale_axes': 0.2,
    'filter_type': 'low',
    'filter_cutoff': 3.0,
    'filter_order': 6,
    'fft_length': 16,
    'do_fft_overlap': True,
    'do_log': True,
    'sampling_freq': 58,
    'raw_samples_per_axis': 116,
}

In [ ]:
def extract_spec_features_axis(data_axis, config):
    fs = config['sampling_freq']
    fft_len = config['fft_length']
    x = data_axis.copy().astype(np.float64)

    # --- Butterworth lowpass (matches C++ apply_filter) ---
    if config['filter_type'] == 'low' and config['filter_order'] > 0:
        sos = signal.butter(config['filter_order'],
                            config['filter_cutoff'],
                            btype='low', fs=fs, output='sos')
        x = signal.sosfilt(sos, x)

    # --- Subtract mean (matches C++ extract_axis_features) ---
    x = x - np.mean(x)

    # --- Time-domain stats (RMS, skewness, kurtosis) ---
    rms = float(np.sqrt(np.mean(x ** 2)))
    std = np.std(x, ddof=0)
    if std == 0:
        std = 1e-10
    skewness = float(np.mean(x ** 3) / (std ** 3))
    kurtosis = float(np.mean(x ** 4) / (std ** 4) - 3)

    # --- PSD max-hold (matches C++ compute_psd_maxhold exactly) ---
    # Custom window copied from anomal_detect.cpp
    window = np.array([0.0] + [0.5] + [1.0] * (fft_len - 3) + [0.5])
    noverlap = fft_len // 2
    _, _, Pxx_all = signal.spectrogram(x, fs=fs, window=window,
                                       nperseg=fft_len,
                                       noverlap=noverlap,
                                       mode='psd')
    Pxx = np.max(Pxx_all, axis=1)  # max-hold across windows

    # --- FFT Skewness / Kurtosis from PSD ---
    pxx_safe = np.maximum(Pxx, 1e-10)
    fft_skew = float(np.mean((pxx_safe - np.mean(pxx_safe)) ** 3) /
                     (np.std(pxx_safe) ** 3 + 1e-10))
    fft_kurt = float(np.mean((pxx_safe - np.mean(pxx_safe)) ** 4) /
                     (np.var(pxx_safe) ** 2 + 1e-10) - 3)

    # --- Log10 of bin 1 only (start_bin=1, stop_bin=2) ---
    val = max(Pxx[1], 1e-10)
    log_val = float(np.log10(val))

    return [rms, skewness, kurtosis, fft_skew, fft_kurt, log_val]


def extract_features(raw_window, config=CONFIG):
    """
    Full DSP pipeline.
    Input:  (116, 3)  raw accelerometer values
    Output: (18,)     feature vector

    Mirrors extract_spectral_analysis_features_v4() → extract_spec_features()
    """
    # --- Scale axes (numpy.hpp:753) ---
    x = raw_window * config['scale_axes']

    features = []
    for axis in range(config['axes']):
        feat = extract_spec_features_axis(x[:, axis], config)
        features.extend(feat)
    return np.array(features, dtype=np.float32)



In [ ]:
# ============================================================
# 2. DATASET PREPARATION (Updated for JSON and windowing)
# ============================================================

def prepare_dataset(raw_data, labels, config=CONFIG):
    """
    raw_data: (n_samples, 116, 3)  numpy array
    labels:   (n_samples,)         integer labels {0,1,2,3}
    returns:  X_features, y
    """
    X = []
    for i in range(len(raw_data)):
        feat = extract_features(raw_data[i], config)
        X.append(feat)
    return np.array(X, dtype=np.float32), np.array(labels)


def load_json_data(dataset_path, config):
    all_raw_windows = [] # Changed to store windows
    all_labels = []

    # 1. Read label mapping from info.labels (now parsing as JSON)
    label_map = {}
    labels_filepath = os.path.join(dataset_path, 'info.labels')
    if not os.path.exists(labels_filepath):
        raise FileNotFoundError(f"info.labels not found at {labels_filepath}")
    with open(labels_filepath, 'r') as f:
        labels_info = json.load(f)

    unique_labels = sorted(list(set(item['label']['label'] for item in labels_info['files'] if 'label' in item and 'label' in item['label'])))
    unique_labels = [l for l in unique_labels if l != 'testing']
    for i, label_name in enumerate(unique_labels):
        label_map[label_name] = i

    if not unique_labels:
        raise ValueError("No unique labels found in info.labels. Check the JSON structure.")

    print(f"Label mapping: {label_map}")

    # Define windowing parameters based on CONFIG
    window_size = config['raw_samples_per_axis']
    stride = window_size // 2 # 50% overlap is common for spectral features

    # Process data from 'training' and 'testing' subdirectories
    subdirs = ['training']
    for subdir in subdirs:
        current_dir = os.path.join(dataset_path, subdir)
        if not os.path.isdir(current_dir):
            print(f"Warning: Directory '{current_dir}' not found. Skipping.")
            continue

        for filename in os.listdir(current_dir):
            if filename.endswith('.json'):
                filepath = os.path.join(current_dir, filename)
                try:
                    with open(filepath, 'r') as f:
                        data = json.load(f)

                    raw_recording_values = data['payload']['values']
                    raw_recording = np.array(raw_recording_values, dtype=np.float32)

                    # Get label from JSON directly, accessing nested 'label' field
                    label_name = data.get('label', {}).get('label')
                    if label_name is None:
                        # Fallback: Extract label from filename
                        match = re.match(r'([a-zA-Z0-9_\-]+)\.json', filename)
                        if match:
                            label_name = match.group(1)

                    if label_name not in label_map:
                        print(f"Warning: Label '{label_name}' from file '{filename}' not found in info.labels or could not be extracted. Skipping entire recording.")
                        continue

                    current_recording_length = raw_recording.shape[0]
                    if current_recording_length < window_size:
                        print(f"Warning: Recording in '{filename}' has length {current_recording_length}, which is less than expected window size {window_size}. Skipping.")
                        continue

                    # Extract windows from the raw recording
                    num_windows_in_recording = (current_recording_length - window_size) // stride + 1

                    for i in range(num_windows_in_recording):
                        start_idx = i * stride
                        end_idx = start_idx + window_size

                        # Ensure the window is exactly the expected size (116, 3)
                        window = raw_recording[start_idx:end_idx]
                        if window.shape == (window_size, config['axes']):
                            all_raw_windows.append(window)
                            all_labels.append(label_map[label_name])
                        else:
                            print(f"Warning: Extracted window from '{filename}' has shape {window.shape}, expected {(window_size, config['axes'])}. Skipping this window.")

                except KeyError as ke:
                    print(f"Error: Missing key in JSON file {filepath} (e.g., 'payload' or 'label'): {ke}. Skipping.")
                except Exception as e:
                    print(f"Error processing file {filepath}: {e}. Skipping.")

    print(f"Loaded {len(all_raw_windows)} valid windows after parsing and windowing.")
    if not all_raw_windows:
        raise ValueError("No data loaded from JSON files after windowing. Check paths, JSON structure, info.labels, and windowing logic.")

    return np.array(all_raw_windows, dtype=np.float32), np.array(all_labels, dtype=np.int32)

In [ ]:
# ============================================================
# 3. MODEL DEFINITION (matches compiled TFLite)
# ============================================================

def build_model(input_dim, num_classes=4):
    """
    Architecture from tflite_learn_1046470_6_compiled.cpp:
      Input:  (1, 18) int8
      FC1:    20 units, ReLU   weights (20x18)
      FC2:    10 units, ReLU   weights (10x20)
      FC3:    4 units, Linear  weights (4x10)
      Softmax
    """
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)), # Explicit Input layer
        tf.keras.layers.Dense(20, activation='relu',
                              name='fc1'),
        tf.keras.layers.Dense(10, activation='relu', name='fc2'),
        tf.keras.layers.Dense(num_classes, activation='softmax', name='fc3'),
    ])
    return model


# ============================================================
# 4. TRAINING
# ============================================================

def train(X_train, y_train, X_val=None, y_val=None, epochs=50, num_classes=4):
    model = build_model(input_dim=X_train.shape[1], num_classes=num_classes) # Pass input_dim
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
    ]

    val_data = (X_val, y_val) if X_val is not None else None
    history = model.fit(
        X_train, y_train,
        validation_data=val_data,
        epochs=epochs,
        batch_size=32,
        callbacks=callbacks,
        verbose=1,
    )
    return model, history


# ============================================================
# 5. STANDARD SCALER (matches C++ ei_data_normalization.h:112)
# ============================================================

def fit_scaler(X_train):
    scaler = StandardScaler(with_mean=True, with_std=True)
    scaler.fit(X_train)
    return scaler


def representative_dataset(X_calib):
    """
    Generator for INT8 quantization calibration.
    """
    for i in range(len(X_calib)):
        feat = X_calib[i]
        yield [feat.reshape(1, -1).astype(np.float32)]


def export_tflite(model, scaler, X_calib, config=CONFIG,
                  filename='model_quantized.tflite', input_dim=18):
    """
    Convert Keras model → INT8 quantized TFLite.
    Mirrors Edge Impulse quantization pipeline.
    """
    # Export the classifier on feature vectors only.
    # DSP extraction must stay outside TFLite because it uses SciPy/Numpy.
    feature_input = tf.keras.layers.Input(shape=(input_dim,), name='feature_input')

    # StandardScaler uses (x - mean) / scale.
    mean = tf.constant(scaler.mean_, dtype=tf.float32)
    scale = tf.constant(scaler.scale_, dtype=tf.float32)
    norm_layer = tf.keras.layers.Lambda(
        lambda x: (x - mean) / scale,
        name='standard_scaler'
    )(feature_input)

    output = model(norm_layer)
    full_model = tf.keras.Model(inputs=feature_input, outputs=output)

    converter = tf.lite.TFLiteConverter.from_keras_model(full_model)

    # Optional: Enable default optimizations (graph transformations etc.)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    # INT8 quantization: representative dataset for calibration
    def rep_gen():
        for i in range(min(200, len(X_calib))):
            feat = X_calib[i]
            yield [feat.reshape(1, -1).astype(np.float32)]
    converter.representative_dataset = rep_gen

    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
        tf.lite.OpsSet.SELECT_TF_OPS
    ]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    tflite_model = converter.convert()

    with open(filename, 'wb') as f:
        f.write(tflite_model)
    print(f"TFLite model saved: {filename} ({len(tflite_model)} bytes)")

    return tflite_model


# ============================================================
# 8. MAIN
# ============================================================

if __name__ == '__main__':
    print("=== Anomaly Detection Training Pipeline ===")
    print("\nLoading dataset: anomaly-detection-export from JSON files..." + os.getcwd())
    dataset_path = '/content/drive/MyDrive/Colab Notebooks/anomaly-detection-export'

    try:
        # Load data using the new JSON parsing function
        raw_data, labels = load_json_data(dataset_path, CONFIG)

        print(f"Raw data shape: {raw_data.shape}")
        print(f"Labels shape: {labels.shape}")

        # Extract features (18-dim)
        print("\nExtracting DSP features...")
        X, y = prepare_dataset(raw_data, labels, CONFIG)
        print(f"Extracted features shape (X): {X.shape}")
        print(f"Labels shape (y): {y.shape}")

        # Train/val split
        print("\nSplitting data into training and validation sets...")
        # Determine the number of classes dynamically from the loaded labels
        num_classes = len(np.unique(y))

        X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
        print(f"Training features shape: {X_tr.shape}")
        print(f"Validation features shape: {X_val.shape}")

        # Fit scaler
        print("\nFitting StandardScaler...")
        scaler = fit_scaler(X_tr)
        print("Transforming training and validation features...")
        X_tr_norm = scaler.transform(X_tr)
        X_val_norm = scaler.transform(X_val)
        print("Features normalized.")

        # Train
        print("\nStarting model training...")
        model, history = train(X_tr_norm, y_tr, X_val_norm, y_val, epochs=50, num_classes=num_classes)
        print("Model training complete.")

        # Export TFLite (with scaler baked in)
        print("\nExporting TFLite model...")
        tflite_model = export_tflite(model, scaler, X_tr, filename='anomaly_detection_model_quantized.tflite', input_dim=X.shape[1])

    except Exception as e:
        print(f"Error during data loading or processing: {e}")
        print("Skipping training and subsequent steps.")
        traceback.print_exc()

In [ ]:
def export_emlearn(model, filename='anomaly_detection_model.h', name='anomaly_model'):
    if emlearn is None:
        raise ImportError(
            "emlearn is not installed. Install it with `pip install emlearn`."
        )

    cmodel = emlearn.convert(model, method='inline')
    cmodel.save(file=filename, name=name)
    print(f"emlearn model saved: {filename}")
    return cmodel

print("\nExporting C/C++ model via emlearn...")
emlearn_model = export_emlearn(model, filename='anomaly_detection_model.h', name='anomaly_model')

print("\n/* Copy these scaler values to anomal_detect.cpp */")
print("static const float NORM_MEAN[FEATURE_LEN] = {", ", ".join(f"{v:.4f}f" for v in scaler.mean_), "};")
print("static const float NORM_SCALE[FEATURE_LEN] = {", ", ".join(f"{v:.6f}f" for v in 1.0 / scaler.scale_), "};")
print(f"\nNum classes: {num_classes}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('training_history.png')
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

y_pred = model.predict(X_val_norm)
y_pred_classes = np.argmax(y_pred, axis=1)

target_names = ['idle', 'left-right', 'maritine', 'up-down']

print('Classification Report (Validation Set):')
print(classification_report(y_val, y_pred_classes, target_names=target_names))

cm = confusion_matrix(y_val, y_pred_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix - Validation')
plt.savefig('confusion_matrix.png')
plt.show()